# ============================================================
# CAPSTONE PROJECT
# NOTEBOOK 24: MULTI-LABEL TARGET MATRIX AND SPLIT AUDIT
# ============================================================
# Purpose:
# This notebook prepares the full multi-label target space for
# genre tagging using the genres_all metadata field.
#
# The goal is to:
# 1. Load the large multi-label master table
# 2. Build full and candidate multi-label target matrices
# 3. Audit label frequencies by split
# 4. Separate core modelling labels from rare-tail labels
# 5. Save clean files for the next modelling phase
# ============================================================

In [1]:
# ============================================================
# 1. IMPORT LIBRARIES
# ============================================================

import os
import ast
import numpy as np
import pandas as pd
from sklearn.preprocessing import MultiLabelBinarizer

In [2]:
# ============================================================
# 2. LOAD SAVED TABLES
# ============================================================

master_df = pd.read_csv("../data/processed/large_multilabel_master_table.csv")
genre_inventory = pd.read_csv("../data/processed/full_genre_inventory.csv")
candidate_genres = pd.read_csv("../data/processed/modelling_candidate_genres.csv")

print("Master table shape:", master_df.shape)
print("Genre inventory shape:", genre_inventory.shape)
print("Candidate genres shape:", candidate_genres.shape)

display(master_df.head())
display(genre_inventory.head())
display(candidate_genres.head())

Master table shape: (81574, 10)
Genre inventory shape: (163, 11)
Candidate genres shape: (150, 11)


,track_id,split,subset,genre_top,title,genres_ids,genres_all_ids,genres_all_names,audio_path,audio_exists
0,20,training,large,NaN,Spiritual Level,"76,103","17,10,76,103",Folk | Pop | Experimental Pop | Singer-Songwriter,../data/raw/audio/fma_large\000\000020.mp3,True
1,26,training,large,NaN,Where is your Love?,"76,103","17,10,76,103",Folk | Pop | Experimental Pop | Singer-Songwriter,../data/raw/audio/fma_large\000\000026.mp3,True
2,30,training,large,NaN,Too Happy,"76,103","17,10,76,103",Folk | Pop | Experimental Pop | Singer-Songwriter,../data/raw/audio/fma_large\000\000030.mp3,True
3,46,training,large,NaN,Yosemite,"76,103","17,10,76,103",Folk | Pop | Experimental Pop | Singer-Songwriter,../data/raw/audio/fma_large\000\000046.mp3,True
4,48,training,large,NaN,Light of Light,"76,103","17,10,76,103",Folk | Pop | Experimental Pop | Singer-Songwriter,../data/raw/audio/fma_large\000\000048.mp3,True


,genre_id,genre_name,parent_id,parent_name,root_genre_id,root_genre_name,top_level_flag,full_direct_count,full_genres_all_count,large_audio_genres_all_count,modelling_tier
0,38,Experimental,NaN,NaN,38,Experimental,38,24912,38154,35903,Tier 1: Strong
1,15,Electronic,NaN,NaN,15,Electronic,15,23866,34413,28099,Tier 1: Strong
2,12,Rock,NaN,NaN,12,Rock,12,8038,32923,25820,Tier 1: Strong
3,1235,Instrumental,NaN,NaN,1235,Instrumental,1235,6055,14938,13588,Tier 1: Strong
4,10,Pop,NaN,NaN,10,Pop,10,6362,13845,12659,Tier 1: Strong


,genre_id,genre_name,parent_id,parent_name,root_genre_id,root_genre_name,top_level_flag,full_direct_count,full_genres_all_count,large_audio_genres_all_count,modelling_tier
0,38,Experimental,NaN,NaN,38,Experimental,38,24912,38154,35903,Tier 1: Strong
1,15,Electronic,NaN,NaN,15,Electronic,15,23866,34413,28099,Tier 1: Strong
2,12,Rock,NaN,NaN,12,Rock,12,8038,32923,25820,Tier 1: Strong
3,1235,Instrumental,NaN,NaN,1235,Instrumental,1235,6055,14938,13588,Tier 1: Strong
4,10,Pop,NaN,NaN,10,Pop,10,6362,13845,12659,Tier 1: Strong


In [3]:
# ============================================================
# 3. PARSE GENRE LIST COLUMNS
# ============================================================

def parse_csv_id_list(value):
    if pd.isna(value):
        return []
    text = str(value).strip()
    if text == "":
        return []
    return [int(x) for x in text.split(",") if str(x).strip() != ""]

master_df["genres_all_id_list"] = master_df["genres_all_ids"].apply(parse_csv_id_list)
master_df["genres_id_list"] = master_df["genres_ids"].apply(parse_csv_id_list)

print("Parsed list columns added.")
display(master_df[["track_id", "genre_top", "genres_all_ids", "genres_all_id_list"]].head(10))

Parsed list columns added.


,track_id,genre_top,genres_all_ids,genres_all_id_list
0,20,NaN,"17,10,76,103","[17, 10, 76, 103]"
1,26,NaN,"17,10,76,103","[17, 10, 76, 103]"
2,30,NaN,"17,10,76,103","[17, 10, 76, 103]"
3,46,NaN,"17,10,76,103","[17, 10, 76, 103]"
4,48,NaN,"17,10,76,103","[17, 10, 76, 103]"
5,135,Rock,"58,12,45","[58, 12, 45]"
6,137,Experimental,"32,1,38","[32, 1, 38]"
7,138,Experimental,"32,1,38","[32, 1, 38]"
8,142,Folk,17,[17]
9,144,Jazz,4,[4]


In [4]:
# ============================================================
# 4. DEFINE FULL AND CANDIDATE LABEL SPACES
# ============================================================

full_label_ids = sorted(genre_inventory["genre_id"].astype(int).tolist())
candidate_label_ids = sorted(candidate_genres["genre_id"].astype(int).tolist())

rare_tail_label_ids = sorted(set(full_label_ids) - set(candidate_label_ids))

print("Total full labels:", len(full_label_ids))
print("Total candidate labels:", len(candidate_label_ids))
print("Total rare-tail labels:", len(rare_tail_label_ids))

print("\nFirst few rare-tail label IDs:", rare_tail_label_ids[:20])

Total full labels: 163
Total candidate labels: 150
Total rare-tail labels: 13

First few rare-tail label IDs: [173, 174, 175, 176, 178, 189, 374, 377, 465, 493, 808, 1032, 1060]


In [5]:
# ============================================================
# 5. BUILD FULL 161-LABEL TARGET MATRIX
# ============================================================

mlb_full = MultiLabelBinarizer(classes=full_label_ids)
Y_full = mlb_full.fit_transform(master_df["genres_all_id_list"])

Y_full_df = pd.DataFrame(
    Y_full,
    columns=[f"genre_{gid}" for gid in mlb_full.classes_],
    index=master_df.index
)

print("Full target matrix shape:", Y_full_df.shape)
display(Y_full_df.head())

Full target matrix shape: (81574, 163)


,genre_1,genre_2,genre_3,genre_4,genre_5,genre_6,genre_7,genre_8,genre_9,genre_10,...,genre_763,genre_808,genre_810,genre_811,genre_906,genre_1032,genre_1060,genre_1156,genre_1193,genre_1235
0,0,0,0,0,0,0,0,0,0,1,...,0,0,0,0,0,0,0,0,0,0
1,0,0,0,0,0,0,0,0,0,1,...,0,0,0,0,0,0,0,0,0,0
2,0,0,0,0,0,0,0,0,0,1,...,0,0,0,0,0,0,0,0,0,0
3,0,0,0,0,0,0,0,0,0,1,...,0,0,0,0,0,0,0,0,0,0
4,0,0,0,0,0,0,0,0,0,1,...,0,0,0,0,0,0,0,0,0,0


In [6]:
# ============================================================
# 6. BUILD 150-LABEL CANDIDATE TARGET MATRIX
# ============================================================

mlb_candidate = MultiLabelBinarizer(classes=candidate_label_ids)
Y_candidate = mlb_candidate.fit_transform(master_df["genres_all_id_list"])

Y_candidate_df = pd.DataFrame(
    Y_candidate,
    columns=[f"genre_{gid}" for gid in mlb_candidate.classes_],
    index=master_df.index
)

print("Candidate target matrix shape:", Y_candidate_df.shape)
display(Y_candidate_df.head())

Candidate target matrix shape: (81574, 150)


c:\Users\jdevo\AppData\Local\Programs\Python\Python39\lib\site-packages\sklearn\preprocessing\_label.py:909: UserWarning: unknown class(es) [1032, 1060, 173, 174, 176, 189, 374, 377, 465, 493, 808] will be ignored
  warnings.warn(


,genre_1,genre_2,genre_3,genre_4,genre_5,genre_6,genre_7,genre_8,genre_9,genre_10,...,genre_693,genre_695,genre_741,genre_763,genre_810,genre_811,genre_906,genre_1156,genre_1193,genre_1235
0,0,0,0,0,0,0,0,0,0,1,...,0,0,0,0,0,0,0,0,0,0
1,0,0,0,0,0,0,0,0,0,1,...,0,0,0,0,0,0,0,0,0,0
2,0,0,0,0,0,0,0,0,0,1,...,0,0,0,0,0,0,0,0,0,0
3,0,0,0,0,0,0,0,0,0,1,...,0,0,0,0,0,0,0,0,0,0
4,0,0,0,0,0,0,0,0,0,1,...,0,0,0,0,0,0,0,0,0,0


In [7]:
# ============================================================
# 7. BUILD LABEL NAME MAPS
# ============================================================

genre_name_map = dict(zip(
    genre_inventory["genre_id"].astype(int),
    genre_inventory["genre_name"].astype(str)
))

candidate_name_map = {gid: genre_name_map.get(gid, f"genre_{gid}") for gid in candidate_label_ids}
rare_tail_name_map = {gid: genre_name_map.get(gid, f"genre_{gid}") for gid in rare_tail_label_ids}

print("Example candidate labels:")
for gid in candidate_label_ids[:10]:
    print(gid, "->", candidate_name_map[gid])

print("\nRare-tail labels:")
for gid in rare_tail_label_ids[:20]:
    print(gid, "->", rare_tail_name_map[gid])

Example candidate labels:
1 -> Avant-Garde
2 -> International
3 -> Blues
4 -> Jazz
5 -> Classical
6 -> Novelty
7 -> Comedy
8 -> Old-Time / Historic
9 -> Country
10 -> Pop

Rare-tail labels:
173 -> N. Indian Traditional
174 -> South Indian Traditional
175 -> Bollywood
176 -> Pacific
178 -> Be-Bop
189 -> Talk Radio
374 -> Banter
377 -> Deep Funk
465 -> Musical Theater
493 -> Western Swing
808 -> Salsa
1032 -> Turkish
1060 -> Tango


In [8]:
# ============================================================
# 8. AUDIT LABEL FREQUENCIES FOR FULL LABEL SPACE
# ============================================================

full_label_counts = Y_full_df.sum(axis=0).sort_values(ascending=False)

full_label_summary = pd.DataFrame({
    "genre_id": [int(col.replace("genre_", "")) for col in full_label_counts.index],
    "genre_name": [genre_name_map[int(col.replace("genre_", ""))] for col in full_label_counts.index],
    "total_count": full_label_counts.values
})

print("Top full-label counts:")
display(full_label_summary.head(30))

Top full-label counts:


,genre_id,genre_name,total_count
0,38,Experimental,35903
1,15,Electronic,28099
2,12,Rock,25820
3,1235,Instrumental,13588
4,10,Pop,12659
5,17,Folk,11187
6,1,Avant-Garde,8134
7,107,Ambient,6799
8,32,Noise,6781
9,76,Experimental Pop,6632


In [9]:
# ============================================================
# 9. AUDIT LABEL FREQUENCIES FOR CANDIDATE LABEL SPACE
# ============================================================

candidate_label_counts = Y_candidate_df.sum(axis=0).sort_values(ascending=False)

candidate_label_summary = pd.DataFrame({
    "genre_id": [int(col.replace("genre_", "")) for col in candidate_label_counts.index],
    "genre_name": [genre_name_map[int(col.replace("genre_", ""))] for col in candidate_label_counts.index],
    "total_count": candidate_label_counts.values
})

print("Top candidate-label counts:")
display(candidate_label_summary.head(30))

Top candidate-label counts:


,genre_id,genre_name,total_count
0,38,Experimental,35903
1,15,Electronic,28099
2,12,Rock,25820
3,1235,Instrumental,13588
4,10,Pop,12659
5,17,Folk,11187
6,1,Avant-Garde,8134
7,107,Ambient,6799
8,32,Noise,6781
9,76,Experimental Pop,6632


In [10]:
# ============================================================
# 10. SPLIT-LEVEL LABEL AUDIT FOR CANDIDATE LABELS
# ============================================================

split_series = master_df["split"].astype(str)

candidate_split_rows = []

for split_name in ["training", "validation", "test"]:
    mask = (split_series == split_name).values
    split_counts = Y_candidate_df.loc[mask].sum(axis=0)

    temp_df = pd.DataFrame({
        "genre_id": [int(col.replace("genre_", "")) for col in split_counts.index],
        "genre_name": [genre_name_map[int(col.replace("genre_", ""))] for col in split_counts.index],
        "split": split_name,
        "count": split_counts.values
    })

    candidate_split_rows.append(temp_df)

candidate_split_summary = pd.concat(candidate_split_rows, ignore_index=True)

print("Candidate split summary shape:", candidate_split_summary.shape)
display(candidate_split_summary.head(30))

Candidate split summary shape: (450, 4)


,genre_id,genre_name,split,count
0,1,Avant-Garde,training,6405
1,2,International,training,3311
2,3,Blues,training,1343
3,4,Jazz,training,2747
4,5,Classical,training,2784
5,6,Novelty,training,612
6,7,Comedy,training,160
7,8,Old-Time / Historic,training,274
8,9,Country,training,1443
9,10,Pop,training,10056


In [11]:
# ============================================================
# 11. LABEL COVERAGE SUMMARY TABLE
# ============================================================

coverage_summary = pd.DataFrame({
    "Label Space": ["Full genres_all", "Candidate modelling labels", "Rare-tail labels"],
    "Number of Labels": [len(full_label_ids), len(candidate_label_ids), len(rare_tail_label_ids)],
    "Description": [
        "All genres present in metadata",
        "Genres with at least 25 occurrences in the large audio subset",
        "Genres below the candidate threshold"
    ]
})

print("Label coverage summary:")
display(coverage_summary)

Label coverage summary:


,Label Space,Number of Labels,Description
0,Full genres_all,163,All genres present in metadata
1,Candidate modelling labels,150,Genres with at least 25 occurrences in the lar...
2,Rare-tail labels,13,Genres below the candidate threshold


In [12]:
# ============================================================
# 12. BUILD MODELLING MASTER TABLES
# ============================================================

multilabel_full_master = pd.concat(
    [master_df[["track_id", "split", "subset", "genre_top", "title", "audio_path", "audio_exists"]], Y_full_df],
    axis=1
)

multilabel_candidate_master = pd.concat(
    [master_df[["track_id", "split", "subset", "genre_top", "title", "audio_path", "audio_exists"]], Y_candidate_df],
    axis=1
)

print("Full multi-label modelling table shape:", multilabel_full_master.shape)
print("Candidate multi-label modelling table shape:", multilabel_candidate_master.shape)

Full multi-label modelling table shape: (81574, 170)
Candidate multi-label modelling table shape: (81574, 157)


In [13]:
# ============================================================
# 13. SAVE OUTPUT FILES
# ============================================================

os.makedirs("../data/processed", exist_ok=True)

full_label_summary.to_csv(
    "../data/processed/full_label_summary.csv",
    index=False
)

candidate_label_summary.to_csv(
    "../data/processed/candidate_label_summary.csv",
    index=False
)

candidate_split_summary.to_csv(
    "../data/processed/candidate_split_summary.csv",
    index=False
)

coverage_summary.to_csv(
    "../data/processed/multilabel_label_coverage_summary.csv",
    index=False
)

multilabel_full_master.to_csv(
    "../data/processed/multilabel_full_master_table.csv",
    index=False
)

multilabel_candidate_master.to_csv(
    "../data/processed/multilabel_candidate_master_table.csv",
    index=False
)

print("Saved:")
print("- ../data/processed/full_label_summary.csv")
print("- ../data/processed/candidate_label_summary.csv")
print("- ../data/processed/candidate_split_summary.csv")
print("- ../data/processed/multilabel_label_coverage_summary.csv")
print("- ../data/processed/multilabel_full_master_table.csv")
print("- ../data/processed/multilabel_candidate_master_table.csv")

Saved:
- ../data/processed/full_label_summary.csv
- ../data/processed/candidate_label_summary.csv
- ../data/processed/candidate_split_summary.csv
- ../data/processed/multilabel_label_coverage_summary.csv
- ../data/processed/multilabel_full_master_table.csv
- ../data/processed/multilabel_candidate_master_table.csv


In [14]:
# ============================================================
# 14. INTERPRETATION NOTES
# ============================================================

print("1. The full metadata target space contains 161 unique genres.")
print("2. A candidate modelling space of 150 labels was created using a minimum-count threshold.")
print("3. The remaining rare-tail labels are still preserved for inventory and future hierarchical handling.")
print("4. The saved multi-label master tables are the basis for full metadata genre identification.")
print("5. The next modelling phase should begin with structured multi-label baselines on the candidate label space.")

1. The full metadata target space contains 161 unique genres.
2. A candidate modelling space of 150 labels was created using a minimum-count threshold.
3. The remaining rare-tail labels are still preserved for inventory and future hierarchical handling.
4. The saved multi-label master tables are the basis for full metadata genre identification.
5. The next modelling phase should begin with structured multi-label baselines on the candidate label space.
